
# ⚡ From Profiling to Parallelism: Where Should My Pipeline Run?

<a href="https://colab.research.google.com/github/rubenfonnegra/HPC/blob/master/Sem_07/From_Profiling_to_Parallelism.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> 
<a href="https://github.com/rubenfonnegra/HPC/tree/master/Sem_07/From_Profiling_to_Parallelism.ipynb" target="_parent"><img src="https://img.shields.io/badge/%E2%80%8B-Open%20in%20Github-blue?logo=github" alt="Open In Github"/></a> 


Last notebook, the pipeline was treated as a single function to answer:

> **Where is my program actually spending time?**

This notebook keeps the same workflow, but now we will try to find out:

> **What parts of my pipeline can run in parallel, and should they run on the CPU or GPU?**


This makes the comparison explicit:

$$
\boxed{
\text{Profile} \rightarrow \text{Hotspot} \rightarrow \text{Parallelism} \rightarrow
\text{CPU/GPU} \rightarrow \text{Measure again}
}
$$

### Learning goals

By the end of the notebook, you should be able to:

- identify the hotspot of a complete computational pipeline;
- distinguish dependencies between stages from parallelism *inside* a stage;
- compare baseline, vectorized, multiprocessing, and GPU implementations;
- quantify CPU↔GPU transfer overhead;
- analyze how workload size changes the best execution strategy;
- justify an architecture decision using **measured evidence**.


---

> **📘 High Performance Computing (HPC)**
> **Author:** Rubén D. Fonnegra, Ph.D. \
> **Institution:** Institución Universitaria Pascual Bravo  
> © 2026 · Educational use with attribution




### Workload

We will use synthetic **high-dimensional scientific/sensor data**:

**Raw data → cleaning → normalization → nonlinear transformation → Gram matrix → eigendecomposition → projection → summary**


## Setup and environment

In [ ]:

import os
import gc
import time
import platform
import warnings
import statistics
import multiprocessing as mp

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import psutil
except ImportError:
    psutil = None

try:
    import torch
except ImportError:
    torch = None

SEED = 42
np.random.seed(SEED)

CPU_COUNT = os.cpu_count() or 1
HAS_FORK = "fork" in mp.get_all_start_methods()
CUDA_AVAILABLE = bool(torch is not None and torch.cuda.is_available())

hardware = {
    "Python": platform.python_version(),
    "OS": platform.platform(),
    "CPU logical cores": CPU_COUNT,
    "RAM (GB)": round(psutil.virtual_memory().total / 1024**3, 2) if psutil else "psutil unavailable",
    "PyTorch": torch.__version__ if torch is not None else "not installed",
    "CUDA available": CUDA_AVAILABLE,
}

if CUDA_AVAILABLE:
    hardware["GPU"] = torch.cuda.get_device_name(0)
    hardware["GPU memory (GB)"] = round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    )

pd.Series(hardware, name="Environment")



## One Pipeline, Four Implementations

We begin by inspecting the machine and generating a dataset with:

- correlated latent structure;
- heterogeneous feature scales;
- additive noise;
- missing values.

The **computational workload remains the same** throughout the notebook.

What changes is only the implementation:

- `run_pipeline_baseline(X)`
- `run_pipeline_vectorized(X)`
- `run_pipeline_multiprocessing(X)`
- `run_pipeline_gpu(X)`


In [ ]:

def make_synthetic_data(
    n_samples=12_000,
    n_features=256,
    latent_dim=12,
    missing_rate=0.01,
    seed=SEED,
):
    """
    Generate correlated, high-dimensional synthetic data.
    """
    rng = np.random.default_rng(seed)

    latent = rng.normal(size=(n_samples, latent_dim)).astype(np.float32)
    loadings = rng.normal(size=(latent_dim, n_features)).astype(np.float32)

    X = latent @ loadings
    X += 0.15 * rng.normal(size=X.shape).astype(np.float32)

    scales = rng.lognormal(mean=0.0, sigma=0.5, size=n_features).astype(np.float32)
    offsets = rng.normal(loc=0.0, scale=2.0, size=n_features).astype(np.float32)
    X = X * scales + offsets

    missing = rng.random(X.shape) < missing_rate
    X[missing] = np.nan

    return X.astype(np.float32)


X = make_synthetic_data()

print(f"Shape: {X.shape}")
print(f"Memory: {X.nbytes / 1024**2:.1f} MB")
print(f"Missing values: {np.isnan(X).mean():.2%}")


In [ ]:

# ---------- Processing functions ----------

def _clean_numpy(X):
    means = np.nanmean(X, axis=0)
    return np.where(np.isnan(X), means, X).astype(np.float32, copy=False)


def _normalize_numpy(X):
    mean = X.mean(axis=0)
    std = X.std(axis=0)
    std = np.where(std < 1e-6, 1.0, std)
    return ((X - mean) / std).astype(np.float32, copy=False)


def _transform_row_loop(X):
    """
    Baseline transformation:
    vectorized arithmetic inside each row, but a Python loop across rows.
    """
    out = np.empty_like(X, dtype=np.float32)

    for i in range(X.shape[0]):
        row = X[i]
        out[i] = np.sin(row) + 0.25 * row * row + np.log1p(np.abs(row))

    return out


def _transform_vectorized(X):
    return (
        np.sin(X) + 0.25 * X * X + np.log1p(np.abs(X))
    ).astype(np.float32, copy=False)


def _transform_chunk(X_chunk):
    return _transform_vectorized(X_chunk)


def _transform_multiprocessing(X, workers=2):
    workers = max(1, int(workers))

    if workers == 1:
        return _transform_vectorized(X)

    if not HAS_FORK:
        warnings.warn(
            "The 'fork' multiprocessing start method is unavailable. "
            "Falling back to vectorized CPU execution."
        )
        return _transform_vectorized(X)

    chunks = [chunk for chunk in np.array_split(X, workers) if len(chunk)]
    ctx = mp.get_context("fork")

    with ctx.Pool(processes=workers) as pool:
        transformed = pool.map(_transform_chunk, chunks)

    return np.vstack(transformed)


def _record_cpu_stage(profile, name, func):
    t0 = time.perf_counter()
    result = func()
    profile[name] = time.perf_counter() - t0
    return result


def _record_gpu_stage(profile, name, func):
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    result = func()
    torch.cuda.synchronize()
    profile[name] = time.perf_counter() - t0
    return result


In [ ]:

# ---------- 1) Baseline pipeline ----------

def run_pipeline_baseline(
    X,
    n_components=16,
    return_profile=False,
):
    profile = {}

    X_clean = _record_cpu_stage(
        profile, "clean", lambda: _clean_numpy(X)
    )

    X_norm = _record_cpu_stage(
        profile, "normalize", lambda: _normalize_numpy(X_clean)
    )

    X_features = _record_cpu_stage(
        profile, "transform", lambda: _transform_row_loop(X_norm)
    )

    gram = _record_cpu_stage(
        profile,
        "gram",
        lambda: (X_features.T @ X_features) / X_features.shape[0],
    )

    def decompose():
        eigenvalues, eigenvectors = np.linalg.eigh(gram)
        order = np.argsort(eigenvalues)[::-1]
        return eigenvectors[:, order[:n_components]]

    components = _record_cpu_stage(
        profile, "decompose", decompose
    )

    projection = _record_cpu_stage(
        profile, "project", lambda: X_features @ components
    )

    summary = _record_cpu_stage(
        profile,
        "summary",
        lambda: np.concatenate(
            [projection.mean(axis=0), projection.std(axis=0)]
        ).astype(np.float32),
    )

    if return_profile:
        return summary, profile

    return summary


In [ ]:

# ---------- 2) Vectorized CPU pipeline ----------

def run_pipeline_vectorized(
    X,
    n_components=16,
    return_profile=False,
):
    profile = {}

    X_clean = _record_cpu_stage(
        profile, "clean", lambda: _clean_numpy(X)
    )

    X_norm = _record_cpu_stage(
        profile, "normalize", lambda: _normalize_numpy(X_clean)
    )

    X_features = _record_cpu_stage(
        profile, "transform", lambda: _transform_vectorized(X_norm)
    )

    gram = _record_cpu_stage(
        profile,
        "gram",
        lambda: (X_features.T @ X_features) / X_features.shape[0],
    )

    def decompose():
        eigenvalues, eigenvectors = np.linalg.eigh(gram)
        order = np.argsort(eigenvalues)[::-1]
        return eigenvectors[:, order[:n_components]]

    components = _record_cpu_stage(
        profile, "decompose", decompose
    )

    projection = _record_cpu_stage(
        profile, "project", lambda: X_features @ components
    )

    summary = _record_cpu_stage(
        profile,
        "summary",
        lambda: np.concatenate(
            [projection.mean(axis=0), projection.std(axis=0)]
        ).astype(np.float32),
    )

    if return_profile:
        return summary, profile

    return summary


In [ ]:

# ---------- 3) Multiprocessing CPU pipeline ----------

def run_pipeline_multiprocessing(
    X,
    workers=None,
    n_components=16,
    return_profile=False,
):
    profile = {}

    if workers is None:
        workers = min(4, CPU_COUNT)

    X_clean = _record_cpu_stage(
        profile, "clean", lambda: _clean_numpy(X)
    )

    X_norm = _record_cpu_stage(
        profile, "normalize", lambda: _normalize_numpy(X_clean)
    )

    X_features = _record_cpu_stage(
        profile,
        "transform",
        lambda: _transform_multiprocessing(X_norm, workers),
    )

    gram = _record_cpu_stage(
        profile,
        "gram",
        lambda: (X_features.T @ X_features) / X_features.shape[0],
    )

    def decompose():
        eigenvalues, eigenvectors = np.linalg.eigh(gram)
        order = np.argsort(eigenvalues)[::-1]
        return eigenvectors[:, order[:n_components]]

    components = _record_cpu_stage(
        profile, "decompose", decompose
    )

    projection = _record_cpu_stage(
        profile, "project", lambda: X_features @ components
    )

    summary = _record_cpu_stage(
        profile,
        "summary",
        lambda: np.concatenate(
            [projection.mean(axis=0), projection.std(axis=0)]
        ).astype(np.float32),
    )

    if return_profile:
        return summary, profile

    return summary


In [ ]:

# ---------- 4) GPU pipeline ----------

def run_pipeline_gpu(
    X,
    n_components=16,
    return_profile=False,
):
    if not CUDA_AVAILABLE:
        raise RuntimeError(
            "CUDA is not available. Enable a GPU runtime to execute this pipeline."
        )

    profile = {}
    device = torch.device("cuda")

    X_gpu = _record_gpu_stage(
        profile,
        "to_device",
        lambda: torch.as_tensor(
            X,
            dtype=torch.float32,
            device=device,
        ),
    )

    def gpu_clean():
        means = torch.nanmean(X_gpu, dim=0)
        return torch.where(torch.isnan(X_gpu), means, X_gpu)

    X_clean = _record_gpu_stage(
        profile, "clean", gpu_clean
    )

    def gpu_normalize():
        mean = X_clean.mean(dim=0)
        std = X_clean.std(dim=0, unbiased=False)
        std = torch.where(std < 1e-6, torch.ones_like(std), std)
        return (X_clean - mean) / std

    X_norm = _record_gpu_stage(
        profile, "normalize", gpu_normalize
    )

    X_features = _record_gpu_stage(
        profile,
        "transform",
        lambda: (
            torch.sin(X_norm)
            + 0.25 * X_norm * X_norm
            + torch.log1p(torch.abs(X_norm))
        ),
    )

    gram = _record_gpu_stage(
        profile,
        "gram",
        lambda: (X_features.T @ X_features) / X_features.shape[0],
    )

    def gpu_decompose():
        eigenvalues, eigenvectors = torch.linalg.eigh(gram)
        return eigenvectors[:, -n_components:]

    components = _record_gpu_stage(
        profile, "decompose", gpu_decompose
    )

    projection = _record_gpu_stage(
        profile, "project", lambda: X_features @ components
    )

    summary_gpu = _record_gpu_stage(
        profile,
        "summary",
        lambda: torch.cat(
            [
                projection.mean(dim=0),
                projection.std(dim=0, unbiased=False),
            ]
        ),
    )

    summary = _record_gpu_stage(
        profile,
        "to_host",
        lambda: summary_gpu.detach().cpu().numpy(),
    ).astype(np.float32)

    if return_profile:
        return summary, profile

    return summary


In [ ]:

baseline_summary = run_pipeline_baseline(X)

print("Baseline pipeline completed.")
print("Summary shape:", baseline_summary.shape)



## Profile First: Where Is the Baseline Spending Time?

Before parallelizing anything, profile the **baseline function**.

Two views are useful:

1. `cProfile` shows where Python spends cumulative time.
2. The internal stage timer shows how much wall-clock time each pipeline stage consumes.

> **Question:** Which stage is actually worth optimizing?


In [ ]:

import cProfile
import pstats
import io

profiler = cProfile.Profile()
profiler.enable()

_ = run_pipeline_baseline(X)

profiler.disable()

stream = io.StringIO()
stats = pstats.Stats(profiler, stream=stream).sort_stats("cumtime")
stats.print_stats(15)

print(stream.getvalue())


In [ ]:

_, baseline_profile = run_pipeline_baseline(
    X,
    return_profile=True,
)

baseline_profile_df = (
    pd.DataFrame(
        {
            "stage": list(baseline_profile.keys()),
            "seconds": list(baseline_profile.values()),
        }
    )
    .assign(percent=lambda d: 100 * d["seconds"] / d["seconds"].sum())
    .sort_values("seconds", ascending=False)
)

display(baseline_profile_df)

ax = baseline_profile_df.sort_values("seconds").plot(
    x="stage",
    y="seconds",
    kind="barh",
    legend=False,
    title="Baseline pipeline — time by stage",
)
ax.set_xlabel("Wall-clock time (s)")
plt.tight_layout()
plt.show()

hotspot = baseline_profile_df.iloc[0]["stage"]
print(f"Measured hotspot: {hotspot}")



### Think before optimizing

- Is the slowest stage parallelizable?
- Is its work independent across **rows**, **columns**, or **operations**?
- Would accelerating a stage that accounts for only a small fraction of runtime matter?
- If we optimize the hotspot, what stage might become the **new** hotspot?



## Map the Parallelism: Dependencies vs. Independent Work

The pipeline stages form a dependency chain:

**clean → normalize → transform → Gram matrix → decompose → project → summary**

So the stages themselves cannot all execute at once.

The more useful question is:

> **What parallelism exists inside each stage?**

| Stage | Dependency between stages? | Parallelism inside the stage | Likely pattern |
|---|---|---|---|
| Cleaning | Yes | Across values/columns | SIMD / vectorization |
| Normalization | Yes | Across values/columns | SIMD / vectorization |
| Transform | Yes | Across rows/elements | SIMD, threads, processes |
| Gram matrix | Yes | Many multiply-adds | Dense linear algebra / GPU |
| Decomposition | Yes | Library-dependent dense algebra | Optimized CPU/GPU library |
| Projection | Yes | Matrix multiplication | Dense linear algebra / GPU |
| Summary | Yes | Parallel reductions | CPU or GPU |



## CPU Experiment: Baseline → Vectorization → Multiprocessing

Now compare three **separate functions**:

```python
run_pipeline_baseline(X)
run_pipeline_vectorized(X)
run_pipeline_multiprocessing(X)
```

This makes the computational changes explicit instead of hiding them behind a mode or strategy parameter.

The goal is not to prove that more workers are always better. Process startup, serialization, copying, and memory bandwidth may erase the benefit of parallel execution.


In [ ]:

def benchmark_function(func, X, repeats=5, warmup=1, **kwargs): 
    """Return median wall-clock time for one pipeline function."""

    for _ in range(warmup):
        _ = func(X, **kwargs)

    times = []
    for _ in range(repeats):
        gc.collect()

        if func is run_pipeline_gpu and CUDA_AVAILABLE:
            torch.cuda.synchronize()

        t0 = time.perf_counter()
        _ = func(X, **kwargs)

        if func is run_pipeline_gpu and CUDA_AVAILABLE:
            torch.cuda.synchronize()

        times.append(time.perf_counter() - t0)

    # return {
    #         "median_s": statistics.median(times),
    #         "mean_s": statistics.mean(times),
    #         "min_s": min(times),
    #         "max_s": max(times),
    #         "all_s": times,
    #     }

    return float(statistics.median(times))


In [ ]:

cpu_rows = []

baseline_time = benchmark_function(
    run_pipeline_baseline,
    X,
    repeat=3,
    warmup=1,
)

cpu_rows.append({
    "implementation": "baseline",
    "workers": 1,
    "seconds": baseline_time,
})

vectorized_time = benchmark_function(
    run_pipeline_vectorized,
    X,
    repeat=3,
    warmup=1,
)

cpu_rows.append({
    "implementation": "vectorized",
    "workers": 1,
    "seconds": vectorized_time,
})

worker_options = sorted(
    set([1, min(2, CPU_COUNT), min(4, CPU_COUNT)])
)

if HAS_FORK:
    for workers in worker_options:
        seconds = benchmark_function(
            run_pipeline_multiprocessing,
            X,
            workers=workers,
            repeat=2,
            warmup=0,
        )

        cpu_rows.append({
            "implementation": "multiprocessing",
            "workers": workers,
            "seconds": seconds,
        })
else:
    print("Multiprocessing experiment skipped: fork start method is unavailable.")

cpu_results = pd.DataFrame(cpu_rows)
cpu_results["speedup_vs_baseline"] = baseline_time / cpu_results["seconds"]

display(cpu_results.sort_values("seconds"))


In [ ]:

vectorized_summary = run_pipeline_vectorized(X)

plot_df = cpu_results.copy()

plot_df["label"] = plot_df.apply(
    lambda r: (
        r["implementation"]
        if r["implementation"] != "multiprocessing"
        else f"mp-{int(r['workers'])}"
    ),
    axis=1,
)

ax = plot_df.plot(
    x="label",
    y="seconds",
    kind="bar",
    legend=False,
    title="CPU pipeline implementations",
)

ax.set_ylabel("Median wall-clock time (s)")
ax.set_xlabel("")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

if HAS_FORK:
    mp_plot = cpu_results.query(
        "implementation == 'multiprocessing'"
    ).sort_values("workers")

    ax = mp_plot.plot(
        x="workers",
        y="speedup_vs_baseline",
        marker="o",
        legend=False,
        title="Multiprocessing speedup",
    )

    ax.set_ylabel("Speedup vs baseline")
    ax.set_xlabel("Processes")
    plt.tight_layout()
    plt.show()



## GPU Experiment: Compute Is Only Part of the Cost

Now compare the CPU implementation with a completely separate GPU function:

```python
run_pipeline_gpu(X)
```

The GPU pipeline measures:

**CPU → GPU transfer → GPU pipeline → GPU → CPU transfer**

This distinction matters because:

> A GPU kernel can be faster while the **complete pipeline** is slower.


In [ ]:

if CUDA_AVAILABLE:
    _ = run_pipeline_gpu(X[:1000])

    gpu_time = benchmark_function(
        run_pipeline_gpu,
        X,
        repeat=3,
        warmup=1,
    )

    gpu_summary, gpu_profile = run_pipeline_gpu(
        X,
        return_profile=True,
    )

    gpu_profile_df = pd.DataFrame(
        {
            "stage": list(gpu_profile.keys()),
            "seconds": list(gpu_profile.values()),
        }
    )

    transfer_time = (
        gpu_profile.get("to_device", 0.0)
        + gpu_profile.get("to_host", 0.0)
    )

    compute_time = sum(
        value
        for key, value in gpu_profile.items()
        if key not in {"to_device", "to_host"}
    )

    gpu_breakdown = pd.DataFrame(
        {
            "component": [
                "CPU→GPU + GPU→CPU transfer",
                "GPU computation",
            ],
            "seconds": [transfer_time, compute_time],
        }
    )

    print(f"Full GPU pipeline median: {gpu_time:.6f} s")

    display(gpu_profile_df)
    display(gpu_breakdown)

    ax = gpu_breakdown.plot(
        x="component",
        y="seconds",
        kind="bar",
        legend=False,
        title="GPU pipeline: transfer vs computation",
    )

    ax.set_ylabel("Time (s)")
    ax.set_xlabel("")
    plt.xticks(rotation=15, ha="right")
    plt.tight_layout()
    plt.show()

else:
    print("CUDA GPU not available.")
    print("Enable a GPU runtime to execute this section.")



### Key question

If GPU **computation** is faster but total GPU **pipeline time** is not, should the workload be moved to the GPU?

Consider:

$$
T_{\text{GPU total}} =
T_{\text{H→D}} +
T_{\text{compute}} +
T_{\text{D→H}}
$$

Also consider whether data could remain on the GPU across multiple operations.



# 6. Scaling Experiment: When Does the Best Implementation Change?

Architecture decisions depend on **problem size**.

A function that loses on a small dataset can win on a larger one because fixed overhead becomes less important.

We will benchmark the same workload with:

- `run_pipeline_baseline`
- `run_pipeline_vectorized`
- `run_pipeline_multiprocessing`
- `run_pipeline_gpu`

> We are looking for a **crossover point**, not a universal winner.


In [ ]:

SCALING_SIZES = [2000, 8000, 20000, 40000]
SCALING_FEATURES = 256
SCALING_WORKERS = min(4, CPU_COUNT)

scaling_rows = []

for n_samples in SCALING_SIZES:
    print(f"Benchmarking n={n_samples} ...")

    X_scale = make_synthetic_data(
        n_samples=n_samples,
        n_features=SCALING_FEATURES,
        seed=SEED + n_samples,
    )

    experiments = [
        ("baseline", run_pipeline_baseline, {}),
        ("vectorized", run_pipeline_vectorized, {}),
    ]

    if HAS_FORK:
        experiments.append(
            (
                "multiprocessing",
                run_pipeline_multiprocessing,
                {"workers": SCALING_WORKERS},
            )
        )

    if CUDA_AVAILABLE:
        experiments.append(
            ("gpu", run_pipeline_gpu, {})
        )

    for name, func, kwargs in experiments:
        seconds = benchmark_function(
            func,
            X_scale,
            repeat=3 if name != "multiprocessing" else 2,
            warmup=1 if name == "gpu" else 0,
            **kwargs,
        )

        row = {
            "n_samples": n_samples,
            "implementation": name,
            "seconds": seconds,
        }

        if name == "gpu":
            _, stage_profile = run_pipeline_gpu(
                X_scale,
                return_profile=True,
            )

            row["gpu_compute_only"] = sum(
                value
                for key, value in stage_profile.items()
                if key not in {"to_device", "to_host"}
            )

        scaling_rows.append(row)

    del X_scale
    gc.collect()

    if CUDA_AVAILABLE:
        torch.cuda.empty_cache()


scaling_df = pd.DataFrame(scaling_rows)
display(scaling_df)


In [ ]:

pivot = scaling_df.pivot(
    index="n_samples",
    columns="implementation",
    values="seconds",
)

display(pivot)

ax = pivot.plot(
    marker="o",
    title="Pipeline runtime vs workload size",
)

ax.set_xlabel("Number of samples")
ax.set_ylabel("Wall-clock time (s)")
plt.tight_layout()
plt.show()

best_by_size = pivot.idxmin(axis=1).rename("fastest_implementation")
best_time = pivot.min(axis=1).rename("seconds")

best_table = pd.concat(
    [best_by_size, best_time],
    axis=1,
)

display(best_table)



# 8. Final Challenge: Build the Best Pipeline

You now have profiling data, CPU benchmarks, GPU timing, transfer costs, and scaling evidence.

Your task is to propose a final implementation while keeping the interface simple:

```python
result = run_pipeline_final(X)
```

You may decide to:

- call the vectorized CPU pipeline;
- call the multiprocessing pipeline;
- call the GPU pipeline;
- write a new hybrid implementation;
- keep some stages on CPU and move others to GPU.

The important part is that the decision must be based on **measurement**, not preference.

## Performance-engineering loop

**Measure → Find hotspot → Understand dependencies → Expose parallelism → Choose architecture → Implement → Measure again**


In [ ]:

# TODO: replace the body after analyzing YOUR benchmark evidence.

def run_pipeline_final(X):
    """
    Final one-function interface.

    Start from the implementation that your measurements justify.
    """
    return run_pipeline_vectorized(X)


final_summary = run_pipeline_final(X)

print("Final result shape:", final_summary.shape)

print(
    "Compatible with baseline:",
    np.allclose(
        np.abs(baseline_summary),
        np.abs(final_summary),
        rtol=5e-3,
        atol=5e-3,
    ),
)


In [ ]:

original_time = benchmark_function(
    run_pipeline_baseline,
    X,
    repeat=3,
    warmup=1,
)

final_time = benchmark_function(
    run_pipeline_final,
    X,
    repeat=3,
    warmup=1,
)

final_speedup = original_time / final_time

final_report = pd.DataFrame(
    {
        "version": [
            "original baseline",
            "final implementation",
        ],
        "seconds": [
            original_time,
            final_time,
        ],
    }
)

display(final_report)

print(f"Final speedup: {final_speedup:.2f}×")


---

## 📄 Attribution

This notebook was developed by **Rubén D. Fonnegra** as educational material for Computación Aplicada (HPC) courses at **Institución Universitaria Pascual Bravo**.
You may use, share, and adapt this material for educational purposes, provided that appropriate credit is given to the original author.

**Suggested citation:**
> Fonnegra Tarazona, R. D. (2026). *Pipeline Performance Engineering: HPC Notebook*. Institución Universitaria Pascual Bravo.